# BiLSTM + Contrastive Learning for Network Traffic Classification
**Team:** Gradient_Issues

Run this notebook on **Google Colab with GPU runtime** (Runtime → Change runtime type → T4 GPU)

Training time: ~2 minutes with GPU (vs 2+ hours on CPU)

In [ ]:
# @title 1. Install dependencies
!pip install -q tensorflow pandas numpy scikit-learn matplotlib seaborn kagglehub

import tensorflow as tf
print(f"TensorFlow: {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")

In [ ]:
# @title 2. Download or generate dataset
import shutil, os, pandas as pd, numpy as np
from pathlib import Path

os.makedirs('data', exist_ok=True)
csv_path = 'data/5g_traffic_classification.csv'

# Try Kaggle download first
downloaded = False
try:
    import kagglehub
    print("Downloading 5G Traffic Dataset from Kaggle...")
    path = kagglehub.dataset_download('kimdaegyeom/5g-traffic-datasets')
    csv_files = list(Path(path).glob('*.csv'))
    if csv_files:
        shutil.copy(csv_files[0], csv_path)
        print(f"Copied {csv_files[0].name} to data/")
        downloaded = True
except Exception as e:
    print(f"Kaggle download failed: {e}")

if not downloaded:
    print("Generating synthetic dataset (100K samples, 6 classes)...")
    np.random.seed(42)
    n = 100000
    class_params = {
        'YouTube':     {'dur': (10,50), 'pkts_fwd': (20,80), 'bytes_fwd': (5000,20000), 'iat_mean': (0.01,0.05), 'pkt_size': (500,1200)},
        'Netflix':     {'dur': (20,80), 'pkts_fwd': (30,120), 'bytes_fwd': (10000,50000), 'iat_mean': (0.02,0.06), 'pkt_size': (800,1400)},
        'Gaming':      {'dur': (30,100), 'pkts_fwd': (10,40), 'bytes_fwd': (500,3000), 'iat_mean': (0.005,0.02), 'pkt_size': (100,400)},
        'Browsing':    {'dur': (2,15), 'pkts_fwd': (5,30), 'bytes_fwd': (1000,8000), 'iat_mean': (0.1,0.5), 'pkt_size': (200,800)},
        'Streaming':   {'dur': (50,200), 'pkts_fwd': (50,200), 'bytes_fwd': (20000,100000), 'iat_mean': (0.01,0.04), 'pkt_size': (1000,1500)},
        'VoIP':        {'dur': (3,20), 'pkts_fwd': (50,150), 'bytes_fwd': (1000,5000), 'iat_mean': (0.01,0.03), 'pkt_size': (100,300)}}
    classes = list(class_params.keys())
    labels = np.random.choice(classes, size=n, p=[0.2,0.2,0.15,0.2,0.15,0.1])
    rows = []
    for lbl in labels:
        p = class_params[lbl]
        dur = np.random.gamma(2, 1) * (p['dur'][1]-p['dur'][0])/2 + p['dur'][0]
        pkts_fwd = int(np.random.gamma(2, 1) * (p['pkts_fwd'][1]-p['pkts_fwd'][0])/2 + p['pkts_fwd'][0])
        pkts_bwd = max(0, int(pkts_fwd * np.random.uniform(0.3, 0.8)))
        bytes_fwd = np.random.gamma(2, 1) * (p['bytes_fwd'][1]-p['bytes_fwd'][0])/2 + p['bytes_fwd'][0]
        bytes_bwd = bytes_fwd * np.random.uniform(0.2, 0.6)
        iat_mean = np.random.gamma(2, 1) * (p['iat_mean'][1]-p['iat_mean'][0])/2 + p['iat_mean'][0]
        rows.append({'duration': dur, 'packets_forward': pkts_fwd, 'bytes_forward': bytes_fwd,
            'packets_backward': pkts_bwd, 'bytes_backward': bytes_bwd,
            'inter_arrival_time_mean': iat_mean, 'inter_arrival_time_std': iat_mean*0.5,
            'packet_size_min': p['pkt_size'][0], 'packet_size_max': p['pkt_size'][1],
            'packet_size_mean': np.mean(p['pkt_size']), 'rtt': np.random.uniform(1, 100),
            'jitter': np.random.uniform(0.1, 10), 'flow_iat_mean': np.random.gamma(2, 1)*0.05,
            'flow_iat_std': np.random.gamma(2, 1)*0.02, 'fwd_iat_mean': iat_mean*np.random.uniform(0.8,1.2),
            'fwd_iat_std': iat_mean*0.3, 'bwd_iat_mean': iat_mean*np.random.uniform(0.8,1.2),
            'bwd_iat_std': iat_mean*0.3, 'protocol_count': np.random.randint(1, 4),
            'src_port_count': np.random.randint(1, 5), 'dst_port_count': np.random.randint(1, 5),
            'syn_flags': np.random.poisson(2), 'fin_flags': np.random.poisson(1),
            'rst_flags': np.random.poisson(0.5), 'label': lbl})
    pd.DataFrame(rows).to_csv(csv_path, index=False)
    print(f"Synthetic dataset saved: {csv_path} ({len(rows)} rows, {len(classes)} classes)")

In [ ]:
# @title 3. Create project files
import os, sys

os.makedirs('models', exist_ok=True)
os.makedirs('scripts', exist_ok=True)
os.makedirs('results', exist_ok=True)

# Make Python find our packages
sys.path.insert(0, '.')

# --- models/encoder.py ---
open('models/encoder.py', 'w').write('''import numpy as np
import tensorflow as tf

class TrafficEncoderBiLSTM(tf.keras.Model):
    def __init__(self, num_classes=5, embedding_dim=32):
        super().__init__()
        self.bilstm_1 = tf.keras.layers.Bidirectional(
            tf.keras.layers.LSTM(64, return_sequences=True, dropout=0.3))
        self.bilstm_2 = tf.keras.layers.Bidirectional(
            tf.keras.layers.LSTM(32, return_sequences=False, dropout=0.3))
        self.dense_1 = tf.keras.layers.Dense(64, activation='relu')
        self.dropout_1 = tf.keras.layers.Dropout(0.3)
        self.embedding_layer = tf.keras.layers.Dense(embedding_dim, name='embedding')
        self.classifier = tf.keras.layers.Dense(num_classes, name='classifier')
        self.num_classes = num_classes
        self.embedding_dim = embedding_dim

    def call(self, inputs, training=False):
        x = tf.expand_dims(inputs, axis=-1)
        x = self.bilstm_1(x, training=training)
        x = self.bilstm_2(x, training=training)
        x = self.dense_1(x, training=training)
        x = self.dropout_1(x, training=training)
        embeddings = self.embedding_layer(x)
        logits = self.classifier(embeddings)
        return logits, embeddings

    @property
    def num_parameters(self):
        return sum(tf.size(w).numpy() for w in self.trainable_weights)

class Trainer:
    def __init__(self, model, optimizer, loss_fn):
        self.model = model
        self.optimizer = optimizer
        self.loss_fn = loss_fn
        self.history = {'train_loss': [], 'val_loss': [], 'val_acc': []}

    @tf.function
    def train_step(self, x, y):
        with tf.GradientTape() as tape:
            logits, _ = self.model(x, training=True)
            loss = self.loss_fn(y, logits)
        grads = tape.gradient(loss, self.model.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.model.trainable_weights))
        return loss

    @tf.function
    def val_step(self, x, y):
        logits, _ = self.model(x, training=False)
        loss = self.loss_fn(y, logits)
        acc = tf.reduce_mean(tf.cast(tf.equal(tf.argmax(logits, 1), y), tf.float32))
        return loss, acc

    def fit(self, train_ds, val_ds, epochs=100, patience=15):
        best_loss = float('inf'); wait = 0; best_ep = 0
        for ep in range(epochs):
            losses = [self.train_step(x, y).numpy() for x, y in train_ds]
            v_loss, v_acc = self.validate(val_ds)
            tl = float(np.mean(losses))
            if v_loss < best_loss:
                best_loss = v_loss; wait = 0; best_ep = ep
                self.model.save_weights('best_model_supervised.weights.h5')
            else:
                wait += 1
            self.history['train_loss'].append(tl)
            self.history['val_loss'].append(v_loss)
            self.history['val_acc'].append(v_acc)
            if (ep + 1) % 10 == 0:
                print(f'Epoch {ep+1}/{epochs} - Loss: {tl:.4f}, Val Loss: {v_loss:.4f}, Val Acc: {v_acc:.4f}')
            if wait >= patience:
                print(f'Early stopping at epoch {ep+1} (best: {best_ep+1})')
                break
        return self.history

    def validate(self, ds):
        losses, accs = [], []
        for x, y in ds:
            l, a = self.val_step(x, y)
            losses.append(l.numpy())
            accs.append(a.numpy())
        return float(np.mean(losses)), float(np.mean(accs))
''')

# --- models/losses.py ---
open('models/losses.py', 'w').write('''import tensorflow as tf

class SupConLoss(tf.keras.losses.Loss):
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature

    def call(self, embeddings, labels):
        embeddings = tf.math.l2_normalize(embeddings, axis=1)
        batch_size = tf.shape(embeddings)[0]
        sim = tf.matmul(embeddings, embeddings, transpose_b=True) / self.temperature
        mask = tf.equal(tf.expand_dims(labels, 1), tf.expand_dims(labels, 0))
        mask = tf.logical_and(mask, ~tf.eye(batch_size, dtype=tf.bool))
        logits = sim - tf.reduce_max(sim, axis=1, keepdims=True)
        log_neg = tf.reduce_logsumexp(logits, axis=1)
        log_pos = tf.math.log(tf.reduce_sum(
            tf.exp(logits) * tf.cast(mask, tf.float32), axis=1) + 1e-6)
        return tf.reduce_mean(-log_pos + log_neg)

class CombinedLoss(tf.keras.losses.Loss):
    def __init__(self, ce_weight=1.0, con_weight=0.5, temperature=0.07):
        super().__init__()
        self.ce_weight = ce_weight
        self.con_weight = con_weight
        self.ce_loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
        self.con_loss = SupConLoss(temperature=temperature)

    def call(self, y_true, logits_and_embeddings):
        logits, embeddings = logits_and_embeddings
        return (self.ce_weight * self.ce_loss(y_true, logits)
                + self.con_weight * self.con_loss(embeddings, y_true))
''')

# --- scripts/preprocess.py ---
open('scripts/preprocess.py', 'w').write('''import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
import tensorflow as tf

class DataLoader:
    FEATURE_COLS = ['duration','packets_forward','bytes_forward','packets_backward',
        'bytes_backward','inter_arrival_time_mean','inter_arrival_time_std',
        'packet_size_min','packet_size_max','packet_size_mean','rtt','jitter',
        'flow_iat_mean','flow_iat_std','fwd_iat_mean','fwd_iat_std',
        'bwd_iat_mean','bwd_iat_std','protocol_count','src_port_count',
        'dst_port_count','syn_flags','fin_flags','rst_flags']

    def __init__(self, csv_path, test_size=0.15, val_size=0.15):
        self.df = pd.read_csv(csv_path).dropna(subset=self.FEATURE_COLS + ['label'])
        self.label_encoder = LabelEncoder()
        self.df['label_enc'] = self.label_encoder.fit_transform(self.df['label'])
        train_df, test_df = train_test_split(self.df, test_size=test_size,
            stratify=self.df['label_enc'], random_state=42)
        adj = val_size / (1 - test_size)
        train_df, val_df = train_test_split(train_df, test_size=adj,
            stratify=train_df['label_enc'], random_state=42)
        self.scaler = StandardScaler()
        self.train_data = (self.scaler.fit_transform(train_df[self.FEATURE_COLS]), train_df['label_enc'].values)
        self.val_data = (self.scaler.transform(val_df[self.FEATURE_COLS]), val_df['label_enc'].values)
        self.test_data = (self.scaler.transform(test_df[self.FEATURE_COLS]), test_df['label_enc'].values)

    def get_tf_dataset(self, X, y, batch_size=64, shuffle=True):
        ds = tf.data.Dataset.from_tensor_slices((X.astype(np.float32), y))
        if shuffle: ds = ds.shuffle(2048)
        return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
''')

# --- __init__.py files (make them proper packages) ---
open('models/__init__.py', 'w').write('from .encoder import TrafficEncoderBiLSTM, Trainer\nfrom .losses import SupConLoss, CombinedLoss\n')
open('scripts/__init__.py', 'w').write('')

print('Project files created!')
print(f'sys.path: {sys.path[:3]}')

In [ ]:
# @title 4. EDA & Preprocessing
from scripts.preprocess import DataLoader

dataloader = DataLoader('data/5g_traffic_classification.csv')
print(f'Dataset: {dataloader.df.shape}')
print(f'Classes: {list(dataloader.label_encoder.classes_)}')
print(f'Train: {len(dataloader.train_data[0])}, Val: {len(dataloader.val_data[0])}, Test: {len(dataloader.test_data[0])}')

import matplotlib.pyplot as plt
dataloader.df['label'].value_counts().plot(kind='bar', title='Class Distribution')
plt.savefig('results/01_class_distribution.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# @title 5. Baseline Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

X_train, y_train = dataloader.train_data
X_test, y_test = dataloader.test_data

for name, clf in [
    ('Logistic Regression', LogisticRegression(max_iter=1000)),
    ('Random Forest', RandomForestClassifier(n_estimators=100))
]:
    clf.fit(X_train, y_train)
    pred = clf.predict(X_test)
    print(f'{name:25s} Acc: {accuracy_score(y_test, pred):.4f}  F1: {f1_score(y_test, pred, average="macro"):.4f}')

In [ ]:
# @title 6. Train Supervised BiLSTM (GPU ~2 min)
from models.encoder import TrafficEncoderBiLSTM, Trainer
import tensorflow as tf

train_ds = dataloader.get_tf_dataset(*dataloader.train_data, shuffle=True)
val_ds = dataloader.get_tf_dataset(*dataloader.val_data, shuffle=False)
test_ds = dataloader.get_tf_dataset(*dataloader.test_data, shuffle=False)

model = TrafficEncoderBiLSTM(num_classes=len(dataloader.label_encoder.classes_), embedding_dim=32)
print(f'Model params: {model.num_parameters:,}')

optimizer = tf.keras.optimizers.Adam(0.001)
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

trainer = Trainer(model, optimizer, loss_fn)
history = trainer.fit(train_ds, val_ds, epochs=100, patience=15)

test_loss, test_acc = trainer.validate(test_ds)
print(f'\nSupervised Test Accuracy: {test_acc:.4f}')

In [ ]:
# @title 7. Train Contrastive BiLSTM (GPU ~3 min)
from models.losses import CombinedLoss

class ContrastiveTrainer(Trainer):
    @tf.function
    def train_step(self, x, y):
        with tf.GradientTape() as tape:
            logits, embeddings = self.model(x, training=True)
            loss = self.loss_fn(y, (logits, embeddings))
        grads = tape.gradient(loss, self.model.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.model.trainable_weights))
        return loss

    @tf.function
    def val_step(self, x, y):
        logits, embeddings = self.model(x, training=False)
        loss = self.loss_fn(y, (logits, embeddings))
        acc = tf.reduce_mean(tf.cast(tf.equal(tf.argmax(logits, 1), y), tf.float32))
        return loss, acc

model_con = TrafficEncoderBiLSTM(num_classes=len(dataloader.label_encoder.classes_), embedding_dim=32)
optimizer_con = tf.keras.optimizers.Adam(0.001)
combined_loss = CombinedLoss(ce_weight=1.0, con_weight=0.5, temperature=0.07)

trainer_con = ContrastiveTrainer(model_con, optimizer_con, combined_loss)
history_con = trainer_con.fit(train_ds, val_ds, epochs=100, patience=15)

test_loss_con, test_acc_con = trainer_con.validate(test_ds)
print(f'\nContrastive Test Accuracy: {test_acc_con:.4f}')

In [ ]:
# @title 8. Final Comparison & Embedding Visualization
import numpy as np
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

# t-SNE on contrastive embeddings
all_emb, all_lbl = [], []
for x, y in test_ds:
    _, emb = model_con(x, training=False)
    all_emb.append(emb.numpy())
    all_lbl.append(y.numpy())
embeddings = np.vstack(all_emb)
labels = np.concatenate(all_lbl)

tsne = TSNE(n_components=2, perplexity=30, random_state=42)
emb_2d = tsne.fit_transform(embeddings[:3000])

plt.figure(figsize=(10, 8))
colors = plt.cm.tab10(np.linspace(0, 1, len(dataloader.label_encoder.classes_)))
for i, name in enumerate(dataloader.label_encoder.classes_):
    mask = labels[:3000] == i
    plt.scatter(emb_2d[mask, 0], emb_2d[mask, 1], c=[colors[i]], label=name, alpha=0.6, s=20)
plt.legend()
plt.title('Contrastive Embeddings (t-SNE)')
plt.savefig('results/tsne_contrastive_embeddings.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n' + '='*50)
print('FINAL RESULTS')
print('='*50)
print(f'BiLSTM Supervised:  {test_acc:.4f}')
print(f'BiLSTM Contrastive: {test_acc_con:.4f}')
print(f'Model params: {model.num_parameters:,}')
print('='*50)

# Download results locally
from google.colab import files
!zip -r results.zip results/ best_model_*.weights.h5
files.download('results.zip')